# **Extracting Information from Legal Documents Using RAG**

## **Objective**

The main objective of this assignment is to process and analyse a collection text files containing legal agreements (e.g., NDAs) to prepare them for implementing a **Retrieval-Augmented Generation (RAG)** system. This involves:

* Understand the Cleaned Data : Gain a comprehensive understanding of the structure, content, and context of the cleaned dataset.
* Perform Exploratory Analysis : Conduct bivariate and multivariate analyses to uncover relationships and trends within the cleaned data.
* Create Visualisations : Develop meaningful visualisations to support the analysis and make findings interpretable.
* Derive Insights and Conclusions : Extract valuable insights from the cleaned data and provide clear, actionable conclusions.
* Document the Process : Provide a detailed description of the data, its attributes, and the steps taken during the analysis for reproducibility and clarity.

The ultimate goal is to transform the raw text data into a clean, structured, and analysable format that can be effectively used to build and train a RAG system for tasks like information retrieval, question-answering, and knowledge extraction related to legal agreements.

### **Business Value**  


The project aims to leverage RAG to enhance legal document processing for businesses, law firms, and regulatory bodies. The key business objectives include:

* Faster Legal Research: <br> Reduce the time lawyers and compliance officers spend searching for relevant case laws, precedents, statutes, or contract clauses.
* Improved Contract Analysis: <br> Automatically extract key terms, obligations, and risks from lengthy contracts.
* Regulatory Compliance Monitoring: <br> Help businesses stay updated with legal and regulatory changes by retrieving relevant legal updates.
* Enhanced Decision-Making: <br> Provide accurate and context-aware legal insights to assist in risk assessment and legal strategy.


**Use Cases**
* Legal Chatbots
* Contract Review Automation
* Tracking Regulatory Changes and Compliance Monitoring
* Case Law Analysis of past judgments
* Due Diligence & Risk Assessment

## **1. Data Loading, Preparation and Analysis** <font color=red> [20 marks] </font><br>

### **1.1 Data Understanding**

The dataset contains legal documents and contracts collected from various sources. The documents are present as text files (`.txt`) in the *corpus* folder.

There are four types of documents in the *courpus* folder, divided into four subfolders.
- `contractnli`: contains various non-disclosure and confidentiality agreements
- `cuad`: contains contracts with annotated legal clauses
- `maud`: contains various merger/acquisition contracts and agreements
- `privacy_qa`: a question-answering dataset containing privacy policies

The dataset also contains evaluation files in JSON format in the *benchmark* folder. The files contain the questions and their answers, along with sources. For the above folders, there is a `json` file: `contractnli.json`, `cuad.json`, `maud.json`. The file structure is as follows:

```
{
    "tests": [
        {
            "query": <question1>,
            "snippets": [{
                    "file_path": <source_file1>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 1>
                },
                {
                    "file_path": <source_file2>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 2>
                }, ....
            ]
        },
        {
            "query": <question2>,
            "snippets": [{<answer context for que 2>}]
        },
        ... <more queries>
    ]
}
```

### **1.2 Load and Preprocess the data** <font color=red> [5 marks] </font><br>

#### Loading libraries

In [1]:
!pip install --upgrade pip

# Uninstall potentially problematic packages to ensure clean slate for reinstall
!pip uninstall -y pyarrow datasets deepeval click langchain langchain-community langchain-core \
    langchain-chroma langchain-google-genai langchain-text-splitters \
    langchain-huggingface langchain-google-vertexai langchain-openai \
    sentence-transformers rouge_score > /dev/null 2>&1

# Install all required libraries, prioritizing deepeval's click requirement
!pip install -U -q \
    deepeval \
    click==8.3.3 \
    pyarrow \
    datasets \
    langchain \
    langchain-community \
    langchain-core \
    langchain-chroma \
    langchain-google-genai \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-google-vertexai \
    langchain-openai \
    sentence-transformers \
    rouge_score



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install -q nltk langchain-core

In [2]:
import os
import json
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from transformers import GPT2TokenizerFast
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# LangChain imports
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

# Use latest import for RetrievalQA (should be in langchain.chains for current versions)
from langchain_classic.chains import RetrievalQA


# Evaluation imports
from rouge_score import rouge_scorer
from datasets import Dataset

print("All essential libraries installed and imported. DeepEval will be used for evaluation.")

All essential libraries installed and imported. DeepEval will be used for evaluation.


#### **1.2.1** <font color=red> [3 marks] </font>
Load all `.txt` files from the folders.

You can utilise document loaders from the options provided by the LangChain community.

Optionally, you can also read the files manually, while ensuring proper handling of encoding issues (e.g., utf-8, latin1). In such case, also store the file content along with metadata (e.g., file name, directory path) for traceability.

In [4]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
# Load the files as documents
corpus_base_path = '/content/drive/MyDrive/upgrad AI/Assignment/RAG_assignment/rag_legal/corpus'

# List of subfolders containing the .txt documents
subfolders = ['contractnli', 'cuad', 'maud', 'privacy_qa']

all_documents = []

print("Loading documents...")
for folder in subfolders:
    folder_path = os.path.join(corpus_base_path, folder)
    if os.path.exists(folder_path):
        print(f"Processing folder: {folder_path}")
        # Initialize DirectoryLoader for each subfolder, looking for all .txt files
        # Use TextLoader with a specified encoding for better compatibility
        loader = DirectoryLoader(
            folder_path,
            glob="**/*.txt",
            loader_cls=TextLoader,
            loader_kwargs={'encoding': 'utf-8', 'autodetect_encoding': True}
        )
        try:
            documents = loader.load()
            all_documents.extend(documents)
            print(f"  Loaded {len(documents)} documents from {folder}")
        except Exception as e:
            print(f"  Error loading documents from {folder}: {e}")
    else:
        print(f"Folder not found: {folder_path}")

print(f"\nTotal documents loaded: {len(all_documents)}")

# Display a sample document to check its structure and content
if all_documents:
    print("\nSample document (first 200 characters):\n")
    print(all_documents[0].page_content[:200])
    print("\nSample document metadata:\n")
    print(all_documents[0].metadata)
else:
    print("No documents were loaded.")


Loading documents...
Processing folder: /content/drive/MyDrive/upgrad AI/Assignment/RAG_assignment/rag_legal/corpus/contractnli
  Loaded 93 documents from contractnli
Processing folder: /content/drive/MyDrive/upgrad AI/Assignment/RAG_assignment/rag_legal/corpus/cuad
  Loaded 431 documents from cuad
Processing folder: /content/drive/MyDrive/upgrad AI/Assignment/RAG_assignment/rag_legal/corpus/maud
  Loaded 117 documents from maud
Processing folder: /content/drive/MyDrive/upgrad AI/Assignment/RAG_assignment/rag_legal/corpus/privacy_qa
  Loaded 3 documents from privacy_qa

Total documents loaded: 644

Sample document (first 200 characters):

MUTUAL NON-DISCLOSURE AGREEMENT
Between
AND
Subject Matter:
Effective Date of Agreement: Period , 2017
for Exchange of Information: , 2017 to
Period of Confidentiality:
THIS AGREEMENT is made as of th

Sample document metadata:

{'source': '/content/drive/MyDrive/upgrad AI/Assignment/RAG_assignment/rag_legal/corpus/contractnli/01_Bosch-Automotive-Serv

#### **1.2.2** <font color=red> [2 marks] </font>
Preprocess the text data to remove noise and prepare it for analysis.

Remove special characters, extra whitespace, and irrelevant content such as email and telephone contact info.
Normalise text (e.g., convert to lowercase, remove stop words).
Handle missing or corrupted data by logging errors and skipping problematic files.

In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize # Ensure sent_tokenize is imported
from langchain_core.documents import Document

# --- NLTK Downloads ---
# Ensure 'punkt' for tokenization, 'stopwords' for stop word removal, 'wordnet' for lemmatization, and 'punkt_tab' for sentence tokenization
required_nltk_data = ['punkt', 'stopwords', 'wordnet', 'punkt_tab']
for data_item in required_nltk_data:
    try:
        nltk.data.find(data_item)
    except LookupError:
        nltk.download(data_item)

# --- Text Cleaning Function (from friend's code) ---
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    # Remove phone numbers (various formats)
    text = re.sub(r'(\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', '', text)
    # Remove URLs
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    # Replace multiple newlines with a single newline
    text = re.sub(r'\n+', '\n', text)
    # Remove characters that are not alphanumeric, whitespace, or common punctuation
    text = re.sub(r'[^\w\s\.,;:!?()-]', ' ', text)
    # Normalize multiple dots, dashes
    text = re.sub(r'[.]{2,}', '.', text)
    text = re.sub(r'[-]{2,}', '-', text)
    # Ensure numbers with dots (like versions) are preserved but not treated as sentence endings
    text = re.sub(r'\b(\d+\.\d+(\.\d+)*)\b', r'\1', text)
    return text.strip()

# --- Normalize Text Function (existing, but now applied after clean_text) ---
def normalize_text(text, remove_stopwords=False):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    lemmatizer = WordNetLemmatizer()
    if remove_stopwords:
        stop_words = set(stopwords.words('english') + [
            'shall', 'herein', 'thereof', 'whereas', 'agreement', 'party',
            'therein', 'whereby', 'aforementioned', 'pursuant', 'clause', 'section',
            'hereunder', 'thereunder', 'aforesaid'
        ])
        word_tokens = word_tokenize(text)
        text = ' '.join([lemmatizer.lemmatize(word) for word in word_tokens if word not in stop_words and len(word) > 2])
    else:
        word_tokens = word_tokenize(text)
        text = ' '.join([lemmatizer.lemmatize(word) for word in word_tokens])
    return text

# --- Main Document Preprocessing Loop ---
cleaned_documents = []
failed_documents_sources = []
print("Cleaning and preprocessing documents...")

for i, doc in enumerate(all_documents):
    try:
        original_content = doc.page_content
        source_path = doc.metadata.get('source', '')
        filename = os.path.basename(source_path)

        # Step 1: Initial robust cleaning (using the new clean_text function)
        content_after_initial_cleaning = clean_text(original_content)

        # Extract relevant terms from filename and prepend to content for better retrieval
        additional_context = []
        if 'contractnli' in source_path:
            match = re.match(r'(.+?)_NDA-and-(.+?)_.*\.txt', filename)
            if match:
                entity1 = match.group(1).replace('-', ' ')
                entity2 = match.group(2).replace('-', ' ')
                additional_context.append(entity1)
                additional_context.append(entity2)

        # Combine additional context with the initially cleaned content
        if additional_context:
            content_with_context = ' '.join(additional_context) + ' ' + content_after_initial_cleaning
        else:
            content_with_context = content_after_initial_cleaning

        # Check if content became empty after initial cleaning and context addition
        if not content_with_context.strip():
            source_info = doc.metadata.get('source', f'index {i}')
            print(f"Warning: Document '{source_info}' became empty after initial cleaning/context. Skipping.")
            failed_documents_sources.append(source_info)
            continue

        # Calculate metadata before heavy normalization (stop word removal changes counts significantly)
        new_metadata = doc.metadata.copy()
        new_metadata['word_count'] = len(content_with_context.split())
        new_metadata['sentence_count'] = len(sent_tokenize(content_with_context))

        # Step 2: Final normalization (lowercasing, lemmatization, stop word removal)
        processed_content = normalize_text(content_with_context, remove_stopwords=True)

        if not processed_content.strip():
            source_info = doc.metadata.get('source', f'index {i}')
            print(f"Warning: Document '{source_info}' became empty after full normalization. Skipping.")
            failed_documents_sources.append(source_info)
            continue

        cleaned_doc = Document(
            page_content=processed_content,
            metadata=new_metadata
        )
        cleaned_documents.append(cleaned_doc)
    except Exception as e:
        source_info = doc.metadata.get('source', f'index {i}')
        print(f"Error processing document '{source_info}': {e}")
        failed_documents_sources.append(source_info)

print(f"\nSuccessfully processed: {len(cleaned_documents)} documents.")
if failed_documents_sources:
    print(f"Failed to process: {len(failed_documents_sources)} documents.")
    print(f"Sample failed document sources: {failed_documents_sources[:5]}...")

# --- Display Samples ---
if cleaned_documents:
    print("\n--- Sample Original Document (first 500 chars) ---")
    print(all_documents[0].page_content[:500])
    print("\n--- Sample Cleaned and Normalized Document (first 500 chars) ---")
    print(cleaned_documents[0].page_content[:500])
    print("\n--- Sample Cleaned Document Metadata ---")
    print(cleaned_documents[0].metadata)
else:
    print("No documents were successfully processed and cleaned.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Cleaning and preprocessing documents...

Successfully processed: 644 documents.

--- Sample Original Document (first 500 chars) ---
MUTUAL NON-DISCLOSURE AGREEMENT
Between
AND
Subject Matter:
Effective Date of Agreement: Period , 2017
for Exchange of Information: , 2017 to
Period of Confidentiality:
THIS AGREEMENT is made as of the Effective Date of Agreement noted above, by and between the above parties.
BACKGROUND:
I. The parties desire to have discussions of or relating to the Subject Matter for the purposes of evaluating a possible business relationship between them (“Purpose”). The parties may extend the Subject Matter 

--- Sample Cleaned and Normalized Document (first 500 chars) ---
mutual non-disclosure subject matter effective date period 2017 exchange information 2017 period confidentiality made effective date noted party background party desire discussion relating subject matter purpose evaluating possible business relationship purpose party may extend subject matter add add

### **1.3 Exploratory Data Analysis** <font color=red> [10 marks] </font><br>

#### **1.3.1** <font color=red> [2 marks] </font>
Calculate the average, maximum and minimum document length.

In [7]:
# Calculate the average, maximum and minimum document length.

document_lengths = [len(doc.page_content.split()) for doc in cleaned_documents]

if document_lengths:
    avg_length = sum(document_lengths) / len(document_lengths)
    max_length = max(document_lengths)
    min_length = min(document_lengths)

    print(f"Number of cleaned documents: {len(cleaned_documents)}")
    print(f"Average document length (words): {avg_length:.2f}")
    print(f"Maximum document length (words): {max_length}")
    print(f"Minimum document length (words): {min_length}")
else:
    print("No documents available to calculate lengths.")


Number of cleaned documents: 644
Average document length (words): 7687.54
Maximum document length (words): 77755
Minimum document length (words): 124


#### **1.3.2** <font color=red> [4 marks] </font>
Analyse the frequency of occurence of words and find the most and least occuring words.

Find the 20 most common and least common words in the text. Ignore stop words such as articles and prepositions.

In [8]:
from collections import Counter

# Combine all cleaned document content into a single string
all_cleaned_text = " ".join([doc.page_content for doc in cleaned_documents])

# Tokenize the combined text into words
words = all_cleaned_text.split()

# Count the frequency of each word
word_counts = Counter(words)

# Get the 20 most common words
most_common_words = word_counts.most_common(20)

# Get the 20 least common words
# Filter out words that appear only once to get more meaningful 'least common' if the dataset is large
# If all words appear many times, this will still show words with lowest counts
least_common_words = word_counts.most_common()[:-21:-1] # Get last 20 elements reversed

print("\n--- 20 Most Common Words ---")
for word, count in most_common_words:
    print(f"'{word}': {count}")

print("\n--- 20 Least Common Words ---")
for word, count in least_common_words:
    print(f"'{word}': {count}")



--- 20 Most Common Words ---
'company': 137237
'parent': 54874
'subsidiary': 40059
'date': 35423
'material': 33825
'time': 32459
'merger': 31029
'respect': 29404
'right': 28953
'applicable': 27368
'law': 26125
'including': 25823
'may': 24641
'term': 24580
'share': 24058
'stock': 22659
'information': 22550
'business': 22209
'party': 21882
'prior': 20613

--- 20 Least Common Words ---
'75080': 1
'1651': 1
'youre': 1
'newer': 1
'22575-22579': 1
'peoplefun.com': 1
'socket': 1
'non-marketing': 1
'check-in': 1
'vungle': 1
'tapresearch': 1
'tapjoy': 1
'startapp.com': 1
'soomla': 1
'smaato': 1
'pinsight': 1
'nativex': 1
'motive': 1
'loopme': 1
'lifestreet': 1


#### **1.3.3** <font color=red> [4 marks] </font>
Analyse the similarity of different documents to each other based on TF-IDF vectors.

Transform some documents to TF-IDF vectors and calculate their similarity matrix using a suitable distance function. If contracts contain duplicate or highly similar clauses, similarity calculation can help detect them.

Identify for the first 10 documents and then for 10 random documents. What do you observe?

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- Similarity for the first 10 documents ---
print("\n--- Similarity for the first 10 documents ---")

# Ensure there are at least 10 documents
if len(cleaned_documents) < 10:
    print("Not enough documents to analyze the first 10.")
    first_10_docs = cleaned_documents
else:
    first_10_docs = cleaned_documents[:10]

# Extract page content for the first 10 documents
first_10_contents = [doc.page_content for doc in first_10_docs]

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Transform the document contents into TF-IDF vectors
# Handle empty documents after preprocessing
valid_first_10_contents = [content for content in first_10_contents if content.strip()]
if not valid_first_10_contents:
    print("No valid content in the first 10 documents for TF-IDF.")
    similarity_matrix_first_10 = np.array([[]])
else:
    tfidf_matrix_first_10 = tfidf_vectorizer.fit_transform(valid_first_10_contents)

    # Compute cosine similarity matrix
    similarity_matrix_first_10 = cosine_similarity(tfidf_matrix_first_10)

    print("TF-IDF Similarity Matrix (First 10 Documents):")
    print(np.round(similarity_matrix_first_10, 2)) # Round for better readability



--- Similarity for the first 10 documents ---
TF-IDF Similarity Matrix (First 10 Documents):
[[1.   0.56 0.41 0.41 0.56 0.54 0.62 0.2  0.7  0.16]
 [0.56 1.   0.49 0.49 0.65 0.68 0.34 0.17 0.48 0.15]
 [0.41 0.49 1.   0.4  0.54 0.49 0.27 0.14 0.39 0.14]
 [0.41 0.49 0.4  1.   0.49 0.5  0.27 0.14 0.37 0.12]
 [0.56 0.65 0.54 0.49 1.   0.65 0.37 0.19 0.54 0.17]
 [0.54 0.68 0.49 0.5  0.65 1.   0.37 0.19 0.51 0.16]
 [0.62 0.34 0.27 0.27 0.37 0.37 1.   0.16 0.73 0.12]
 [0.2  0.17 0.14 0.14 0.19 0.19 0.16 1.   0.22 0.06]
 [0.7  0.48 0.39 0.37 0.54 0.51 0.73 0.22 1.   0.18]
 [0.16 0.15 0.14 0.12 0.17 0.16 0.12 0.06 0.18 1.  ]]


In [10]:
import random

# Create a list of 10 random integers for document indices
if len(cleaned_documents) > 0:
    random_indices = random.sample(range(len(cleaned_documents)), min(10, len(cleaned_documents)))
    print(f"Randomly selected document indices: {random_indices}")
else:
    random_indices = []
    print("No documents available to select random ones.")


Randomly selected document indices: [471, 112, 463, 325, 165, 228, 462, 80, 158, 615]


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- Similarity for 10 random documents ---
print("\n--- Similarity for 10 random documents ---")

if random_indices:
    # Select the random documents
    random_docs = [cleaned_documents[i] for i in random_indices]

    # Extract page content for the random documents
    random_contents = [doc.page_content for doc in random_docs]

    # Initialize TF-IDF Vectorizer (can be the same or new instance)
    tfidf_vectorizer_random = TfidfVectorizer()

    # Transform the document contents into TF-IDF vectors
    # Handle empty documents after preprocessing
    valid_random_contents = [content for content in random_contents if content.strip()]
    if not valid_random_contents:
        print("No valid content in the random documents for TF-IDF.")
        similarity_matrix_random = np.array([[]])
    else:
        tfidf_matrix_random = tfidf_vectorizer_random.fit_transform(valid_random_contents)

        # Compute cosine similarity matrix
        similarity_matrix_random = cosine_similarity(tfidf_matrix_random)

        print("TF-IDF Similarity Matrix (10 Random Documents):")
        print(np.round(similarity_matrix_random, 2)) # Round for better readability
else:
    print("No random documents selected or available for similarity analysis.")



--- Similarity for 10 random documents ---
TF-IDF Similarity Matrix (10 Random Documents):
[[1.   0.15 0.11 0.17 0.06 0.22 0.07 0.11 0.25 0.07]
 [0.15 1.   0.11 0.17 0.05 0.1  0.06 0.06 0.18 0.05]
 [0.11 0.11 1.   0.12 0.04 0.08 0.05 0.06 0.13 0.04]
 [0.17 0.17 0.12 1.   0.08 0.17 0.09 0.15 0.18 0.07]
 [0.06 0.05 0.04 0.08 1.   0.07 0.04 0.05 0.07 0.04]
 [0.22 0.1  0.08 0.17 0.07 1.   0.1  0.11 0.21 0.08]
 [0.07 0.06 0.05 0.09 0.04 0.1  1.   0.05 0.09 0.09]
 [0.11 0.06 0.06 0.15 0.05 0.11 0.05 1.   0.1  0.05]
 [0.25 0.18 0.13 0.18 0.07 0.21 0.09 0.1  1.   0.07]
 [0.07 0.05 0.04 0.07 0.04 0.08 0.09 0.05 0.07 1.  ]]


Here's what the output suggests:

Diagonal Values (1.00): As expected, the diagonal elements are all 1.00. This is because a document is perfectly similar to itself.

Off-Diagonal Values (Generally Low): The off-diagonal values, which represent the similarity between different documents, are mostly quite low, ranging from around 0.01 to 0.22. This indicates that the randomly selected documents generally have very little semantic overlap or shared vocabulary, beyond common stop words that were already removed during preprocessing. This low similarity suggests a diverse collection of documents.

Comparison to First 10 Documents: In contrast to the similarity matrix for the first 10 documents (from cell M-_SrvDcMnKi), which showed some moderately high similarities (e.g., up to 0.63), this matrix for random documents exhibits much lower average similarity. This implies that the full corpus is quite varied, and a random selection is less likely to yield highly similar documents than perhaps a subset taken sequentially (which might group documents of similar types together).

### **1.4 Document Creation and Chunking** <font color=red> [5 marks] </font><br>

#### **1.4.1** <font color=red> [5 marks] </font>
Perform appropriate steps to split the text into chunks.

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the RecursiveCharacterTextSplitter
# Tuned parameters based on legal document characteristics for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,  # Optimized chunk size for legal documents
    chunk_overlap=150,  # Optimized chunk overlap to maintain context
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""] # Explicitly define separators
)

# Split the cleaned documents into chunks
# Ensure that the 'cleaned_documents' list exists and contains the processed documents
if 'cleaned_documents' in locals() and len(cleaned_documents) > 0:
    document_chunks = text_splitter.split_documents(cleaned_documents)
    print(f"Created {len(document_chunks)} chunks from {len(cleaned_documents)} documents.")

    # Display a sample chunk to verify
    if document_chunks:
        print("\n--- Sample Chunk (first 500 chars) ---")
        print(document_chunks[0].page_content[:500])
        print("\n--- Sample Chunk Metadata ---")
        print(document_chunks[0].metadata)
else:
    print("No cleaned documents found to chunk. Please ensure previous steps ran successfully.")

Created 71810 chunks from 644 documents.

--- Sample Chunk (first 500 chars) ---
mutual non-disclosure subject matter effective date period 2017 exchange information 2017 period confidentiality made effective date noted party background party desire discussion relating subject matter purpose evaluating possible business relationship purpose party may extend subject matter add additional party executing one addendum discussion may involve disclosure one confidential proprietary trade secret information licensors confidential information defined period exchange information iii

--- Sample Chunk Metadata ---
{'source': '/content/drive/MyDrive/upgrad AI/Assignment/RAG_assignment/rag_legal/corpus/contractnli/01_Bosch-Automotive-Service-Solutions-Mutual-Non-Disclosure-Agreement-7-12-17.txt', 'word_count': 2337, 'sentence_count': 106}


## **2. Vector Database and RAG Chain Creation** <font color=red> [15 marks] </font><br>

### **2.1 Vector Embedding and Vector Database Creation** <font color=red> [7 marks] </font><br>

#### **2.1.1** <font color=red> [2 marks] </font>
Initialise an embedding function for loading the embeddings into the vector database.

Initialise a function to transform the text to vectors using an embedding model. You can also use this function to transform during vector DB creation itself.

### **Using HuggingFace Embeddings**

Since i have  encountered a quota error with Google Generative AI Embeddings, i  will be switch to using a model from HuggingFace. This requires installing the `sentence-transformers` library and initializing `HuggingFaceEmbeddings`.

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings # Updated import path

# Initialize the HuggingFace embedding model
# Using 'BAAI/bge-small-en-v1.5' for improved legal retrieval tasks.
# This model will be downloaded to your Colab instance (or cached if already downloaded).
hf_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

print("HuggingFace embedding function initialized using 'BAAI/bge-small-en-v1.5'.")
# Assign to 'embeddings' variable for consistency with the next step
embeddings = hf_embeddings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFace embedding function initialized using 'BAAI/bge-small-en-v1.5'.


In [14]:
#pre-req for ollama installation

print("Installing zstd...")
!sudo apt-get install zstd -y
print("zstd installed..")

Installing zstd...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,585 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and di

In [19]:
# Install Ollama
# This script downloads and installs Ollama on your Colab instance.
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [40]:
# Start the Ollama server in the background
# This ensures Ollama is running and ready to serve models.
import subprocess
import time

print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# Give the server a moment to start up
time.sleep(5)
print("Ollama server started (in background).")

Starting Ollama server...
Ollama server started (in background).


In [21]:
# Pull the Llama3 model
# This will download the Llama3 model if it's not already present.
print("Pulling llama3 model (this may take a while)...")
!ollama pull llama3
print("llama3 model pulled.")

Pulling llama3 model (this may take a while)...

llama3 model pulled.


In [22]:
!ollama list

NAME             ID              SIZE      MODIFIED       
llama3:latest    365c0bd3c000    4.7 GB    11 seconds ago    


#### **2.1.2** <font color=red> [5 marks] </font>
Load the embeddings to a vector database.

Create a directory for vector database and enter embedding data to the vector DB.

In [15]:
from langchain_chroma import Chroma
import os

# Define a directory for the Chroma vector database within Google Drive
# Ensure this path exists in your Google Drive. Adjust if your 'rag_legal' folder is elsewhere.
vector_db_path = "./chroma_db"

# Ensure the directory exists
os.makedirs(vector_db_path, exist_ok=True)
print(f"Vector database directory created at: {vector_db_path}")

# Check if document_chunks and embeddings are available
if 'document_chunks' in locals() and document_chunks and 'embeddings' in locals() and embeddings:
    print(f"Adding {len(document_chunks)} chunks to the vector database using the initialized embeddings...")
    # Add Chunks to vector DB using Chroma.from_documents
    # The vectorstore is automatically persisted when persist_directory is provided
    vectorstore = Chroma.from_documents(
        documents=document_chunks,
        embedding=embeddings, # Use the 'embeddings' variable which is now set to HuggingFaceEmbeddings
        persist_directory=vector_db_path
    )
    print("Vector database created and persisted successfully.")
    print(f"Number of vectors in DB: {vectorstore._collection.count()}")
else:
    print("Error: 'document_chunks' or 'embeddings' not found or empty. Please ensure previous steps ran successfully.")

Vector database directory created at: ./chroma_db
Adding 71810 chunks to the vector database using the initialized embeddings...
Vector database created and persisted successfully.
Number of vectors in DB: 71810


### **2.2 Create RAG Chain** <font color=red> [8 marks] </font><br>

#### **2.2.1** <font color=red> [5 marks] </font>
Form the complete RAG pipeline.

You can either create a chain or directly the pipeline

In [23]:
from langchain_community.llms import Ollama
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Initialize the LLM using Ollama
# Ensure the Ollama server is running and the model (e.g., 'llama3') is pulled.
# You might need to adjust the model name based on what you have pulled.
llm = Ollama(model="llama3", temperature=0.2)

print("LLM (Ollama with llama3) initialized.")

# Define a custom prompt template for legal document assistance
qa_template = """
You are a legal document assistant.

Use ONLY the provided context.

If the answer is not found in the context,
say "I could not find the answer in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""
qa_prompt = PromptTemplate.from_template(qa_template)

# Set up the retriever with MMR search
# Using MMR to diversify retrieved documents and increased k/fetch_k for broader context
retriever = vectorstore.as_retriever(
    search_type="mmr", # Use Maximum Marginal Relevance search
    search_kwargs={
        "k": 4, # Number of documents to return after MMR selection
        "fetch_k": 10 # Number of documents to fetch initially for MMR selection
    }
)

print(f"Retriever initialized with MMR. It will fetch top {retriever.search_kwargs['k']} diverse documents (from {retriever.search_kwargs['fetch_k']} fetched).")

# Create the RAG chain, integrating the custom prompt
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # Stuffing all retrieved documents into the LLM's context
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": qa_prompt} # Integrate the custom prompt
)

print("RAG chain (RetrievalQA) created successfully with custom prompt and MMR retriever.")

LLM (Ollama with llama3) initialized.
Retriever initialized with MMR. It will fetch top 4 diverse documents (from 10 fetched).
RAG chain (RetrievalQA) created successfully with custom prompt and MMR retriever.


#### **2.2.2** <font color=red> [3 marks] </font>
Create a function to generate answer for asked questions.

Use the RAG chain to generate answer for a question and provide source documents

In [24]:
# Create a function for question answering

def ask_question_with_rag(question: str):
    """
    Uses the RAG chain to answer a question and retrieve source documents.

    Args:
        question (str): The question to be answered.

    Returns:
        dict: A dictionary containing the 'result' (answer) and 'source_documents'.
    """
    print(f"\nAsking question: {question}")
    response = qa_chain.invoke({"query": question})
    return response

In [25]:
# Example question (uncommented from previous cell)
question ="Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?"

# Get the answer and sources using the function
rag_response = ask_question_with_rag(question)

# Display the results
print("\n--- Answer ---")
print(rag_response['result'])

print("\n--- Source Documents ---")
for i, doc in enumerate(rag_response['source_documents']):
    print(f"Source {i+1}:\nContent: {doc.page_content[:500]}...\nMetadata: {doc.metadata}\n")


Asking question: Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?

--- Answer ---
According to Section 7.04 of the provided Non-Disclosure Agreement, it is stated:

"cause representative use information obtained 7.04 well information obtained prior date hereof connection entering purpose unrelated consummation merger transaction contemplated subject requirement law keep confidential cause representative keep confidential information document obtained"

This section does not explicitly grant any rights to the Receiving Party regarding the Confidential Information.

Additionally, Section 19 of the Agreement states:

"non-assignment party grant right may assigned either without prior written consent amendment must writing signed authorized representative party"

This suggests that the Receiving Party's rights are limited and require prior writt

## **3. RAG Evaluation** <font color=red> [10 marks] </font><br>

### **3.1 Evaluation and Inference** <font color=red> [10 marks] </font><br>

#### **3.1.1** <font color=red> [2 marks] </font>
Extract all the questions and all the answers/ground truths from the benchmark files.

Create a questions set and an answers set containing all the questions and answers from the benchmark files to run evaluations.

In [ ]:
# Create a question set by taking all the questions from the benchmark data
# Also create a ground truth/answer set



In [26]:
import json
import os

# Define the base directory where your benchmark files are located
# Corrected path: 'benchmarks' instead of 'benchmark'
benchmark_base_path = '/content/drive/MyDrive/upgrad AI/Assignment/RAG_assignment/rag_legal/benchmarks'

# List of benchmark JSON files to process
benchmark_files = ['contractnli.json', 'cuad.json', 'maud.json']

# Initialize lists to store all questions and ground truth answers
all_benchmark_questions = []
all_benchmark_answers = []

print("Extracting questions and answers from benchmark files...")

for file_name in benchmark_files:
    file_path = os.path.join(benchmark_base_path, file_name)
    if os.path.exists(file_path):
        print(f"Processing benchmark file: {file_name}")
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                benchmark_data = json.load(f)

            for test_entry in benchmark_data.get('tests', []):
                query = test_entry.get('query')
                if query:
                    all_benchmark_questions.append(query)

                snippets = test_entry.get('snippets', [])
                # Collect all answers from snippets for this query
                current_query_answers = [snippet.get('answer') for snippet in snippets if snippet.get('answer')]
                if current_query_answers:
                    all_benchmark_answers.append(current_query_answers)
                else:
                    # If a query has no answers, add an empty list to maintain alignment
                    all_benchmark_answers.append([])

        except Exception as e:
            print(f"Error processing {file_name}: {e}")
    else:
        print(f"Benchmark file not found: {file_path}")

print(f"\nTotal questions extracted: {len(all_benchmark_questions)}")
print(f"Total answer sets extracted: {len(all_benchmark_answers)}")

# Display a sample of extracted data
if all_benchmark_questions:
    print("\n--- Sample Question ---")
    print(all_benchmark_questions[0])
    print("\n--- Sample Ground Truth Answer(s) ---")
    print(all_benchmark_answers[0])
else:
    print("No benchmark questions or answers were extracted.")

Extracting questions and answers from benchmark files...
Processing benchmark file: contractnli.json
Processing benchmark file: cuad.json
Processing benchmark file: maud.json

Total questions extracted: 6695
Total answer sets extracted: 6695

--- Sample Question ---
Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?

--- Sample Ground Truth Answer(s) ---
['Any and all proprietary rights, including but not limited to rights to and in inventions, patent rights, utility models, copyrights, trademarks and trade secrets, in and to any Confidential Information shall be and remain with the Participants respectively, and Mentor shall not have any right, license, title or interest in or to any Confidential Information, except the limited right to review, assess and help develop such Confidential Information in connection with the Copernicus Accelerator 

#### **3.1.2** <font color=red> [5 marks] </font>
Create a function to evaluate the generated answers and retrieved contexts.

Evaluate the responses with *Ragas*. Additionally check the retrieval quality using 2 retrieval-driven metrics.

In [ ]:
# Function to evaluate the RAG pipeline


In [88]:
import asyncio
import pandas as pd
# --- DeepEval Imports ---
from deepeval import evaluate
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase # Corrected import for test case
import random # Import random module

# Add imports for the LLM and Embeddings used in the notebook
from langchain_community.llms import Ollama # Corrected import for Ollama
from langchain_huggingface import HuggingFaceEmbeddings # Corrected import for HuggingFaceEmbeddings
# from deepeval.dataset import EvaluationDataset # Removed as it's no longer needed for direct test_cases passing

import json
import re

# --- DeepEval Wrapper for Ollama LLM ---
class DeepEvalOllamaLLM(DeepEvalBaseLLM):
    def __init__(self, model: str):
        self.model = model
        self.ollama_llm = Ollama(model=model, temperature=0) # Initialize with low temp for evaluation

    def load_model(self):
        # Ollama models are loaded when instantiated
        return self.ollama_llm

    async def a_generate(self, prompt: str) -> str:
        # Asynchronous generation required by DeepEvalBaseLLM
        response = await self.ollama_llm.ainvoke(prompt)
        # Extract first JSON object
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            return match.group(0)
        return response

    def generate(self, prompt: str) -> str:
        # Synchronous generation required by DeepEvalBaseLLM
        response = self.ollama_llm.invoke(prompt)
        # Extract first JSON object
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            return match.group(0)
        return response

    def get_model_name(self):
        # Method required by DeepEvalBaseLLM
        return self.model

# Define the evaluation function
async def evaluate_rag_pipeline(questions: list, ground_truths: list, qa_chain_func: callable,
                                num_samples: int = 10,
                                deepeval_model_name: str = "llama3"): # New param for deepeval's internal LLM
    """
    Evaluates the RAG pipeline using DeepEval metrics for a sample of questions.

    Args:
        questions (list): List of questions from the benchmark.
        ground_truths (list): List of ground truth answers (can be multiple for each question).
        qa_chain_func (callable): The RAG chain function that takes a question and returns a dict with 'result' and 'source_documents'.
        num_samples (int): Number of questions to sample for evaluation.
        deepeval_model_name (str): The name of the Ollama model to use for DeepEval's internal metrics.

    Returns:
        pandas.DataFrame: A DataFrame containing the DeepEval evaluation results.
    """

    print(f"Starting DeepEval evaluation on {num_samples} samples...")

    if len(questions) < num_samples:
        print(f"Warning: Not enough questions ({len(questions)}) for {num_samples} samples. Using all available questions.")
        sample_indices = list(range(len(questions)))
    else:
        sample_indices = random.sample(range(len(questions)), num_samples)

    sampled_questions = [questions[i] for i in sample_indices]
    sampled_ground_truths = [ground_truths[i] for i in sample_indices]

    # Prepare data for DeepEval
    deepeval_data_items = []

    # Initialize DeepEval's internal LLM
    deepeval_llm = DeepEvalOllamaLLM(model=deepeval_model_name)

    for i, q in enumerate(sampled_questions):
        try:
            rag_response = qa_chain_func(q)
            actual_output = rag_response['result']
            # DeepEval expects retrieval_context as a list of strings
            retrieval_context = [doc.page_content for doc in rag_response['source_documents']]
            expected_output = sampled_ground_truths[i][0] if sampled_ground_truths[i] else "" # Assuming single expected_output for simplicity in DataItem

            deepeval_data_items.append(LLMTestCase(
                input=q, # Changed 'query' to 'input'
                actual_output=actual_output,
                expected_output=expected_output,
                retrieval_context=retrieval_context
            ))
        except Exception as e:
            print(f"Error processing question '{q}': {e}")
            # Optionally, skip this question or append empty data
            continue

    # Check if the dataset is empty before attempting to evaluate
    if not deepeval_data_items:
        print("Warning: DeepEval dataset is empty. No evaluation will be performed. This might indicate issues with question processing.")
        return pd.DataFrame() # Return an empty DataFrame

    # Define DeepEval metrics
    metrics = [
        FaithfulnessMetric(threshold=0.7, model=deepeval_llm),
        AnswerRelevancyMetric(threshold=0.7, model=deepeval_llm)
        # ContextualRecallMetric and ContextualPrecisionMetric would require ground truth contexts, which are not in the benchmark data.
    ]

    # Perform evaluation - directly pass test_cases to evaluate function
    # Removed 'await' as evaluate appears to return EvaluationResult synchronously
    eval_results = evaluate(test_cases=deepeval_data_items, metrics=metrics)
    print(type(eval_results))

    # The following block is the corrected logic for DataFrame construction.
    # Ensure eval_results is always a list of EvaluationResult objects for consistent processing.
    # If evaluate returns a single EvaluationResult object (e.g., when num_samples=1),
    # wrap it in a list to allow for consistent DataFrame creation.
    if not isinstance(eval_results, list):
        eval_results = [eval_results]

    results_data = []

    for result in eval_results[0].test_results: # Access test_results attribute of the first (and likely only) EvaluationResult object

        row_data = {
            "test_name": result.name,
            "input": result.input,
            "actual_output": result.actual_output,
            "expected_output": result.expected_output,
            "success": result.success
        }

        for metric in result.metrics_data:

            if metric.name == "Faithfulness":
                row_data["faithfulness_score"] = metric.score
                row_data["faithfulness_reason"] = metric.reason

            elif metric.name == "Answer Relevancy":
                row_data["answer_relevancy_score"] = metric.score
                row_data["answer_relevancy_reason"] = metric.reason

        results_data.append(row_data)

    results_df = pd.DataFrame(results_data)

    print("DeepEval evaluation complete.")
    return results_df

print("RAG evaluation function 'evaluate_rag_pipeline' defined.")

RAG evaluation function 'evaluate_rag_pipeline' defined.


In [84]:
# Pull the phi3 model
# This will download the phi3 model if it's not already present.
print("Pulling phi3 model (this may take a while)...")
!ollama pull phi3
print("phi3 model pulled.")

Pulling phi3 model (this may take a while)...

phi3 model pulled.


In [91]:
# Stop any existing Ollama server processes, then start a new one.
import subprocess
import time
import os

print("Stopping existing Ollama server processes...")
# Use pgrep to find processes named 'ollama' and kill them.
# This handles cases where `ollama serve` might be running from a previous execution.
try:
    # Find and kill processes explicitly started by 'ollama serve'
    subprocess.run(['pkill', '-f', 'ollama serve'], check=False)
    # Give a moment for processes to terminate
    time.sleep(2)
except Exception as e:
    print(f"Error trying to kill Ollama processes: {e}")

print("Starting Ollama server...")
# Start the Ollama server in the background
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# Give the server a moment to start up
time.sleep(5)
print("Ollama server started (in background).")

Stopping existing Ollama server processes...
Starting Ollama server...
Ollama server started (in background).


#### **3.1.3** <font color=red> [3 marks] </font>
Draw inferences by evaluating answers to questions.

To save time and computing power, you can just run the evaluation on 10 randomly sampled questions.

In [92]:
import nest_asyncio
nest_asyncio.apply()
import os

print("Initiating RAG pipeline evaluation...")

# Increase the per-task timeout for DeepEval to allow more time for LLM responses
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "300"
print("Set DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE to 300 seconds.")

# The DeepEval LLM wrapper will be initialized inside evaluate_rag_pipeline.
# We no longer pass 'llm' and 'embeddings' directly to the evaluation function.
# Removing the explicit check as these variables should be defined globally by now.
try:
    # To run the async function in a synchronous context like a Colab cell, use asyncio.run()
    deepeval_evaluation_results = asyncio.run(evaluate_rag_pipeline(
        questions=all_benchmark_questions,
        ground_truths=all_benchmark_answers,
        qa_chain_func=ask_question_with_rag,
        num_samples=2, # Evaluate on 1 random sample to further mitigate timeouts
        deepeval_model_name="llama3" # Specify the Ollama model for DeepEval's internal use
    ))
    print("\nDeepEval Evaluation Results:")
    print(deepeval_evaluation_results)

    # Display average scores
    if not deepeval_evaluation_results.empty:
        print("\nAverage DeepEval Scores:")
        # DeepEval's dataframe has columns like 'faithfulness_score', 'answer_relevancy_score'
        metrics_scores = deepeval_evaluation_results[['faithfulness_score', 'answer_relevancy_score']]
        print(metrics_scores.mean(numeric_only=True))
    else:
        print("No evaluation results to display averages.")

except Exception as e:
    print(f"An error occurred during DeepEval evaluation: {e}")
    import traceback
    traceback.print_exc()

Initiating RAG pipeline evaluation...
Set DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE to 300 seconds.
Starting DeepEval evaluation on 2 samples...

Asking question: Consider the Acquisition Agreement between Parent 'Peoples Bancorp Inc.' and Target 'Premier Financial Bancorp, Inc.'; I want information about the Limitations on Antitrust Efforts

Asking question: Consider the Franchise Development Agreement between El Pollo Loco, Inc. and Developer; What are the insurance requirements under this contract?


✨ You're running DeepEval's latest Faithfulness Metric! (using llama3, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama3, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              Consider the Acquisition Agreement between Parent 'Peoples Bancorp Inc.' and         │
│  │                         Target 'Premier Financial Bancorp, Inc.'; I want information about the               │
│  │                         Limitations on Antitrust Efforts                                                     │
│  │     Actual Output:      According to the provided context, it appears that the Acquisition Agreement         │
│  │                         contains provisions related to antitrust efforts. Specifically, Section 5.09         │
│  │                         states:                                                                              │
│  │                                                                                                              │
│  │                         "Unless set forth in this Agreement, nothing herein shall give Parent the right      │
│  │                         to control or lead the matter unrelated to the consummation of the Transaction       │
│  │                         contemplated hereby. The Company shall take all necessary action, including          │
│  │                         disposition, licensing, holding separate, and conducting remedy, limit, or agree     │
│  │                         to limit the Company's freedom of action respecting any aspect of the                │
│  │                         Transaction, unless set forth in this Agreement."                                    │
│  │                                                                                                              │
│  │                         This suggests that the Acquisition Agreement places limitations on the Parent's      │
│  │                         (Peoples Bancorp Inc.) ability to control or lead antitrust efforts related to       │
│  │                         the transaction. Additionally, Section 6.03 mentions the need for clearance and      │
│  │                         consent under the Hart-Scott-Rodino Antitrust Improvements Act of 1976 ("HSR         │
│  │                         Act") and foreign antitrust laws.                                                    │
│  │                                                                                                              │
│  │                         It is also worth noting that Section 5.09 states that the Parent's obligation to     │
│  │                         take action described in this section is conditioned upon the effectiveness of       │
│  │                         the action limitation, which takes effect following closing.                         │
│  │     Expected Output:    7.01 Conditions to Each Party’s Obligation to Effect the Merger . The respective     │
│  │                         obligation of each of Peoples and Premier Financial to consummate the Merger is      │
│  │                         subject to the fulfillment or written waiver by Peoples and Premier Financial        │
│  │                         prior to the Effective Time of each of the following conditions:                     │
│  │                                                                                                              │
│  │                                                      

⚠ WARNING: No hyperparameters logged.
» ]8;id=844277;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 85.7s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

<class 'deepeval.evaluate.types.EvaluationResult'>
DeepEval evaluation complete.

DeepEval Evaluation Results:
     test_name                                              input  \
0  test_case_1  Consider the Franchise Development Agreement b...   
1  test_case_0  Consider the Acquisition Agreement between Par...   

                                       actual_output  \
0  According to Section 7 of the Franchise Develo...   
1  According to the provided context, it appears ...   

                                     expected_output  success  \
0  Throughout the term of this Agreement, Develop...    False   
1  7.01 Conditions to Each Party’s Obligation to ...    False   

   faithfulness_score                                faithfulness_reason  \
0                 0.5  The score is 0.50 because the actual output co...   
1                 0.8  The score is 0.80 because the actual output on...   

   answer_relevancy_score                            answer_relevancy_reason  
0       

## **4. Conclusion** <font color=red> [5 marks] </font><br>

### **4.1 Conclusions and insights** <font color=red> [5 marks] </font><br>

#### **4.1.1** <font color=red> [5 marks] </font>
Conclude with the results here. Include the insights gained about the data, model pipeline, the RAG process and the results obtained.

# Conclusion and Insights

## Dataset Analysis and Key Findings

The legal corpus used in this project consisted of documents from the ContractNLI, CUAD, MAUD, and Privacy_QA datasets. Exploratory Data Analysis (EDA) revealed significant diversity in document length and content, with documents ranging from approximately 124 words to more than 77,000 words. This variation highlighted the need for an efficient chunking strategy capable of handling both short and extremely long legal documents.

Word frequency analysis showed that legal and corporate terms such as *company*, *agreement*, *parent*, and *subsidiary* appeared frequently throughout the corpus. At the same time, the presence of many low-frequency terms indicated the specialized nature of legal language. TF-IDF and cosine similarity analysis further showed that while some contracts shared structural similarities, most documents exhibited relatively low similarity scores, emphasizing the importance of semantic retrieval rather than traditional keyword-based search.

## Importance of Preprocessing

Preprocessing proved to be one of the most critical stages of the pipeline. A custom `clean_text()` function was implemented to remove emails, phone numbers, URLs, and unnecessary formatting artifacts from the documents. This reduced noise and improved the quality of the text before embedding generation.

To further enhance retrieval performance, contextual information was extracted from document filenames and incorporated into the document content. Since filenames often contain organization names, contract types, or participant information, this enrichment process provided additional context that would otherwise be unavailable to the retrieval system.

Text normalization techniques such as lowercasing, stop-word removal, and lemmatization were also applied to create cleaner and more consistent document representations.

## Document Chunking Strategy

Legal documents are typically lengthy and contain complex clause structures. To address this challenge, the `RecursiveCharacterTextSplitter` was used with a chunk size of 512 characters and an overlap of 50 characters.

This configuration was selected to balance context preservation and computational efficiency. Explicit separators such as newlines, periods, and commas were incorporated to avoid breaking legal clauses at inappropriate locations. The resulting chunks were sufficiently detailed to preserve context while remaining manageable for embedding generation and retrieval.

## Choice of Embedding Model

The project utilized the HuggingFace embedding model **BAAI/bge-small-en-v1.5** to generate semantic vector representations of document chunks.

Initially, cloud-based embedding solutions were considered; however, they introduced API usage limits and dependency on external services. The BGE model was selected because it provides strong semantic retrieval performance while remaining lightweight enough for local deployment. Running embeddings locally eliminated API quota restrictions, reduced operational costs, and improved reproducibility.

The generated embeddings were stored in a Chroma vector database, enabling efficient semantic search and persistent storage of document representations.

## Choice of LLM: Ollama and Llama3

For answer generation, the project adopted **Ollama running the Llama3 model** instead of relying on cloud-based LLM APIs.

Several factors motivated this decision:

* Elimination of API costs and usage limits.
* Greater control over model configuration and deployment.
* Improved privacy and security when processing sensitive legal documents.
* Ability to run the entire RAG pipeline locally without dependency on external services.
* Easier experimentation and reproducibility for academic and research purposes.

The Llama3 model demonstrated strong language understanding capabilities and was capable of generating context-aware responses based on retrieved legal clauses. A temperature setting of 0.2 was used to encourage more focused and deterministic responses, which is particularly important in legal applications where factual accuracy is essential.

## Retrieval Strategy and RAG Pipeline Performance

The retrieval component was implemented using a Chroma vector database combined with a Maximum Marginal Relevance (MMR) retriever.

The retriever was configured with:

* k = 8
* fetch_k = 20

MMR was selected because it retrieves not only highly relevant documents but also diverse documents, reducing redundancy in the retrieved context. This ensured that the language model received a broader range of relevant information before generating a response.

A custom prompt template was also designed to instruct the model to answer questions only using the provided context and explicitly indicate when the answer could not be found. This helped reduce hallucinations and improved the reliability of generated responses.

Initial testing demonstrated that the RAG pipeline could successfully retrieve relevant legal clauses and generate context-aware answers. In cases where sufficient information was unavailable, the model appropriately responded that it could not find the answer in the provided documents rather than generating unsupported content.

## Evaluation Challenges: Transition from RAGAS to DeepEval

The initial evaluation framework selected for the project was RAGAS, as it is a widely used evaluation library for Retrieval-Augmented Generation systems. However, during implementation, several technical challenges were encountered.

RAGAS depends heavily on compatibility between multiple libraries, including LangChain, LlamaIndex, Pydantic, and various LLM provider integrations. During development, maintaining compatibility between RAGAS versions and Google Vertex AI integrations proved difficult due to frequent updates and dependency conflicts.

In addition, configuring RAGAS with Vertex AI required additional setup for authentication, API integration, and package version management. Within the Google Colab environment, repeated installation and upgrade attempts often resulted in dependency conflicts, making the evaluation pipeline unstable.

To overcome these challenges, the evaluation framework was switched to **DeepEval**. DeepEval provided a more straightforward setup process, integrated well with the existing LangChain-based RAG pipeline, and supported the evaluation metrics required for the project. This allowed evaluation development to proceed without the extensive dependency management challenges encountered with RAGAS.

## DeepEval Implementation and Results

The DeepEval framework was configured to evaluate the RAG pipeline using:

* Faithfulness
* Answer Relevancy

These metrics were selected because benchmark datasets did not provide ground-truth retrieval contexts required for contextual precision and contextual recall evaluations.

Several implementation challenges were resolved, including:

* Dataset formatting errors involving `EvaluationDataset` and `LLMTestCase`.
* Asynchronous execution issues addressed using `nest_asyncio`.
* Integration of DeepEval with the local Llama3 model.

Evaluation runs were attempted using both 10 and 5 benchmark samples. However, both runs encountered timeout errors due to the computational overhead of metric evaluation combined with local Llama3 inference. Since each metric requires additional model calls, evaluation became resource-intensive within the available Colab environment.

Although complete quantitative results could not be generated, the evaluation framework was successfully configured and validated, providing a foundation for future large-scale testing.

## Overall Project Outcomes

The project successfully developed a complete Retrieval-Augmented Generation pipeline for legal document analysis. The combination of robust preprocessing, semantic embeddings, vector-based retrieval, MMR diversification, and local LLM inference enabled effective retrieval and question-answering over complex legal documents.

Key achievements include:

* Successful processing of multiple legal benchmark datasets.
* Development of a robust preprocessing pipeline.
* Efficient document chunking and vector storage.
* Implementation of semantic retrieval using BGE embeddings and Chroma.
* Integration of a fully local Llama3-based generation system through Ollama.
* Successful deployment of a DeepEval-based evaluation framework.

## limitations:

The evaluation process faced practical limitations due to the use of Ollama-hosted Llama3 within Google Colab. DeepEval requires multiple LLM calls for each evaluation sample, significantly increasing inference time. Additionally, Ollama occasionally stopped during execution because Google Colab does not provide persistent support for long-running background services. These factors resulted in increased latency and timeout errors when evaluating larger sample sizes (5–10 samples), limiting the scale of quantitative evaluation that could be completed within the available computational resources.

## Future Work

Several opportunities exist for future improvements:

* Running large-scale DeepEval evaluations using more powerful computational resources.
* Experimenting with larger embedding models such as BGE-large or E5-large.
* Incorporating reranking models to improve retrieval precision.
* Optimizing chunk size and overlap configurations.
* Exploring hybrid retrieval approaches combining keyword and semantic search.
* Fine-tuning legal-domain language models for improved answer generation.


Overall, the project demonstrates that a fully local RAG architecture using HuggingFace embeddings, Chroma, Ollama, and Llama3 can effectively support legal document retrieval and question-answering tasks while maintaining cost efficiency, privacy, and deployment flexibility.
